<a href="https://colab.research.google.com/github/carlduya-tech/Prompt-Engineering/blob/main/Carl_Duya_Prompt_Engineering_Exercise_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
Customer Support Bot (no API keys)
- Multi-step internal flow (classify -> gather -> propose -> escalate) WITHOUT announcing steps
- 3+ randomized response variants per option so users see different phrasing each time
"""

from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, Optional, Tuple, Any
import random


# -----------------------------
# Fake data layer (in-memory)
# -----------------------------
@dataclass
class Order:
    order_id: str
    name: str
    status: str          # "Processing", "Shipped", "Out for delivery", "Delivered", "Cancelled"
    carrier: str
    tracking_number: str
    eta: datetime
    delivered_at: Optional[datetime] = None
    return_eligible_until: Optional[datetime] = None


def seed_orders() -> Dict[str, Order]:
    now = datetime.now()
    return {
        "A1001": Order(
            order_id="A1001",
            name="Jordan Lee",
            status="Shipped",
            carrier="UPS",
            tracking_number="1Z999AA10123456784",
            eta=now + timedelta(days=2),
            return_eligible_until=now + timedelta(days=30),
        ),
        "A1002": Order(
            order_id="A1002",
            name="Sam Patel",
            status="Out for delivery",
            carrier="FedEx",
            tracking_number="449044304137821",
            eta=now + timedelta(hours=8),
            return_eligible_until=now + timedelta(days=30),
        ),
        "A1003": Order(
            order_id="A1003",
            name="Avery Chen",
            status="Delivered",
            carrier="USPS",
            tracking_number="9400111899223847182736",
            eta=now - timedelta(days=1),
            delivered_at=now - timedelta(days=1, hours=3),
            return_eligible_until=now + timedelta(days=14),
        ),
        "A1004": Order(
            order_id="A1004",
            name="Taylor Rivera",
            status="Processing",
            carrier="—",
            tracking_number="—",
            eta=now + timedelta(days=4),
            return_eligible_until=now + timedelta(days=30),
        ),
    }


# -----------------------------
# Utilities
# -----------------------------
def now() -> datetime:
    return datetime.now()


def pick(variants) -> str:
    """Randomly pick one response variant."""
    return random.choice(list(variants))


def prompt_nonempty(prompt: str) -> str:
    while True:
        v = input(prompt).strip()
        if v:
            return v
        print("Please enter a value.\n")


def prompt_yes_no(prompt: str) -> bool:
    while True:
        v = input(prompt).strip().lower()
        if v in {"y", "yes"}:
            return True
        if v in {"n", "no"}:
            return False
        print("Please answer y/n.\n")


def prompt_choice(prompt: str, options: Dict[str, str]) -> str:
    print(prompt)
    for k, label in options.items():
        print(f"{k}) {label}")
    while True:
        v = input("Select: ").strip()
        if v in options:
            return v
        print("Invalid choice.\n")


def prompt_order_id() -> str:
    return prompt_nonempty("Enter your Order ID (e.g., A1001): ").upper()


def get_order(orders: Dict[str, Order], order_id: str) -> Optional[Order]:
    return orders.get(order_id)


def format_order_summary(order: Order) -> str:
    lines = [
        "--- Order Summary ---",
        f"Order ID:         {order.order_id}",
        f"Name:             {order.name}",
        f"Status:           {order.status}",
        f"Carrier:          {order.carrier}",
        f"Tracking Number:  {order.tracking_number}",
        f"ETA:              {order.eta.strftime('%Y-%m-%d %H:%M')}",
    ]
    if order.delivered_at:
        lines.append(f"Delivered At:     {order.delivered_at.strftime('%Y-%m-%d %H:%M')}")
    if order.return_eligible_until:
        lines.append(f"Return Eligible:  Until {order.return_eligible_until.strftime('%Y-%m-%d')}")
    lines.append("---------------------")
    return "\n".join(lines)


def make_ticket(prefix: str, order_id: str) -> str:
    return f"{prefix}-{order_id}-{now().strftime('%Y%m%d-%H%M%S')}"


# -----------------------------
# Randomized response templates (3+ each)
# -----------------------------
RESPONSES = {
    "greeting": [
        "Hi—how can I help today?",
        "Hello! What can I assist you with?",
        "Welcome. What do you need help with?"
    ],
    "order_not_found": [
        "I can’t find that order ID. Let’s grab a few details so I can locate it.",
        "That order ID isn’t showing up on my side—let me collect some info to look it up.",
        "I’m not seeing that order ID. Share a couple details and I’ll help track it down."
    ],
    "invalid_option": [
        "That option doesn’t look right—please pick a number from the list.",
        "Oops, I didn’t get that. Choose one of the menu numbers.",
        "Try again using one of the listed options."
    ],
    "track_intro": [
        "Here are the latest tracking details I have:",
        "Here’s what I’m seeing for your tracking info:",
        "This is the current tracking snapshot:"
    ],
    "delayed_not_past_eta": [
        "Good news: it’s still within the ETA window. Carrier scans can be a bit laggy—check again soon.",
        "It hasn’t passed the ETA yet. I’d keep an eye on tracking—updates often land later in the day.",
        "You’re still within the estimated window. If it slips past the ETA, I can escalate it."
    ],
    "delayed_past_eta": [
        "It’s past the ETA, so the next best step is to open a carrier investigation.",
        "Since the ETA has passed, I recommend starting a delivery investigation with the carrier.",
        "This looks late. I can open an investigation so the carrier can locate the package."
    ],
    "missing_checks_needed": [
        "Before we open a case, please check nearby spots (porch/side door/mail room) and ask neighbors/household.",
        "Quick checklist: look around the delivery area, ask neighbors/household, and check any front desk/mail room.",
        "First step: check around the property and with neighbors/household. If it’s still missing, we’ll open a case."
    ],
    "missing_wait_24h": [
        "Since it hasn’t been 24 hours yet, I recommend waiting a bit—sometimes deliveries are scanned early.",
        "If it was just marked delivered, give it up to 24 hours—packages occasionally show up late.",
        "I’d wait until the 24-hour mark; if it’s still missing after that, we’ll file a missing-package case."
    ],
    "missing_escalate": [
        "You’ve checked and waited 24 hours—let’s open a missing-package case.",
        "Okay, that meets the criteria to escalate. I’ll open a missing-package case.",
        "Thanks—since you’ve done the checks and waited 24 hours, I’ll create a missing-package case."
    ],
    "return_not_delivered": [
        "Returns usually require delivery first. Once it’s delivered, I can start the return for you.",
        "This order isn’t marked delivered yet. After it arrives, we can initiate a return.",
        "I can’t start a return until delivery is confirmed. Come back once it’s delivered."
    ],
    "return_expired": [
        "It looks like the return window has expired for this order.",
        "I’m seeing that the return eligibility period has passed.",
        "This order appears outside the return window."
    ],
    "return_created": [
        "All set—your return is started. I’ve created an RMA and a return label (simulated).",
        "Done. I’ve initiated your return and generated the RMA/label (simulated).",
        "Return initiated—RMA created and label generated (simulated)."
    ],
    "address_change_blocked": [
        "Address changes aren’t available once an order is out for delivery or delivered.",
        "I can’t update the address at this stage (out for delivery/delivered).",
        "This order is too far along for an address change."
    ],
    "address_change_ok": [
        "Got it. I’ve recorded the address change request (simulated).",
        "Okay—address change request captured (simulated).",
        "Done. I’ve noted the new address for this order (simulated)."
    ],
    "cancel_blocked": [
        "I can’t cancel because the order has already shipped. If it arrives, we can do a return.",
        "This order is already in transit, so cancellation isn’t available. A return is the next best option.",
        "Cancellation isn’t possible after shipping, but you can return it once delivered."
    ],
    "cancelled": [
        "Done—your order is cancelled.",
        "Confirmed. I’ve cancelled the order.",
        "All set—order cancelled."
    ],
    "no_changes": [
        "Okay—no changes applied.",
        "No problem. I won’t make any changes.",
        "Got it—leaving everything as-is."
    ],
    "escalation_created": [
        "I’ve created a case and passed the details to the next team.",
        "Case created—this has been escalated for follow-up.",
        "I opened a case so the team can investigate further."
    ],
}


# -----------------------------
# Menu / classification
# -----------------------------
ISSUES = {
    "1": "Track order",
    "2": "Delivery delayed",
    "3": "Missing order",
    "4": "Return order",
    "5": "Change delivery address",
    "6": "Cancel order",
    "0": "Exit",
}


def classify_issue() -> str:
    print(pick(RESPONSES["greeting"]))
    return prompt_choice("Select a support topic:", ISSUES)


# -----------------------------
# Gather info (dynamic)
# -----------------------------
def gather_info(issue_key: str, orders: Dict[str, Order]) -> Tuple[Optional[Order], Dict[str, Any]]:
    ctx: Dict[str, Any] = {}
    if issue_key == "0":
        return None, ctx

    order_id = prompt_order_id()
    ctx["order_id"] = order_id
    order = get_order(orders, order_id)

    if not order:
        print(pick(RESPONSES["order_not_found"]))
        ctx["found_order"] = False
        ctx["customer_name"] = prompt_nonempty("Name on the order: ")
        ctx["email"] = prompt_nonempty("Email used at checkout: ")
        if issue_key in {"2", "3"}:
            ctx["delivery_zip"] = prompt_nonempty("Delivery ZIP/postal code: ")
        return None, ctx

    ctx["found_order"] = True
    print()
    print(format_order_summary(order))
    print()

    # Issue-specific questions (only when needed)
    if issue_key == "2":  # Delivery delayed
        ctx["eta_passed"] = now() > order.eta
        if not ctx["eta_passed"]:
            ctx["wants_updates"] = prompt_yes_no("Want proactive updates until the ETA? (y/n): ")
        else:
            ctx["package_moved_recently"] = prompt_yes_no("Has tracking shown movement in the last 48 hours? (y/n): ")
            ctx["delivery_notes"] = prompt_nonempty("Any delivery notes (gate code, access info, etc.)? ")

    elif issue_key == "3":  # Missing order
        ctx["where_delivered"] = prompt_nonempty("Where does the carrier say it was delivered (porch, mailbox, front desk)? ")
        ctx["checked_surroundings"] = prompt_yes_no("Have you checked nearby areas / neighbors / mail room? (y/n): ")
        if order.status.lower() == "delivered":
            ctx["waited_24h"] = prompt_yes_no("Has it been at least 24 hours since it was marked delivered? (y/n): ")

    elif issue_key == "4":  # Return order
        ctx["return_reason"] = prompt_nonempty("Return reason (wrong size, damaged, changed mind, etc.): ")
        ctx["opened"] = prompt_yes_no("Was the item opened/used? (y/n): ")

    elif issue_key == "5":  # Change delivery address
        ctx["new_address"] = prompt_nonempty("Enter the new delivery address (single line): ")

    elif issue_key == "6":  # Cancel order
        ctx["confirm_cancel"] = prompt_yes_no("Do you want to cancel this order? (y/n): ")

    return order, ctx


# -----------------------------
# Propose solution (uses gathered info)
# -----------------------------
def propose_solution(issue_key: str, order: Optional[Order], ctx: Dict[str, Any]) -> Dict[str, Any]:
    # If order wasn't found -> needs escalation (manual lookup)
    if not ctx.get("found_order", False):
        return {
            "outcome": "needs_escalation",
            "recommended_action": "Open manual lookup case",
            "message": (
                "I can open a manual lookup case using your name/email.\n"
                "If we still can’t locate it, we’ll treat it as a missing-order claim."
            ),
            "ticket_prefix": "LOOKUP",
        }

    assert order is not None

    if issue_key == "1":  # Track
        intro = pick(RESPONSES["track_intro"])
        msg = (
            f"{intro}\n"
            f"- Carrier: {order.carrier}\n"
            f"- Tracking: {order.tracking_number}\n"
            f"- Status: {order.status}\n"
            f"- ETA: {order.eta.strftime('%Y-%m-%d %H:%M')}"
        )
        return {"outcome": "resolved", "recommended_action": "Check carrier tracking", "message": msg}

    if issue_key == "2":  # Delivery delayed
        if order.status.lower() == "delivered":
            return {
                "outcome": "cannot_process",
                "recommended_action": "Use Missing order",
                "message": "This order shows as delivered. If you can’t find it, choose “Missing order”.",
            }

        if not ctx.get("eta_passed", False):
            msg = pick(RESPONSES["delayed_not_past_eta"])
            action = "Enable proactive updates" if ctx.get("wants_updates") else "Monitor until ETA"
            return {"outcome": "resolved", "recommended_action": action, "message": msg}

        # ETA passed -> escalate to carrier investigation
        msg = pick(RESPONSES["delayed_past_eta"])
        # Include signal from gathered info (movement + notes) so it meaningfully uses prior output
        moved = "Yes" if ctx.get("package_moved_recently") else "No"
        notes = ctx.get("delivery_notes", "").strip()
        msg += f"\n\nDetails captured:\n- Movement in last 48h: {moved}\n- Delivery notes: {notes or 'None'}"
        return {
            "outcome": "needs_escalation",
            "recommended_action": "Open carrier investigation",
            "message": msg,
            "ticket_prefix": "DELAY",
        }

    if issue_key == "3":  # Missing order
        if order.status.lower() != "delivered":
            return {
                "outcome": "cannot_process",
                "recommended_action": "Track / Delivery delayed",
                "message": "This order isn’t marked delivered yet. Please use tracking/delay support first.",
            }

        where = ctx.get("where_delivered", "unknown")
        checked = bool(ctx.get("checked_surroundings"))
        waited = bool(ctx.get("waited_24h"))

        if not checked:
            msg = pick(RESPONSES["missing_checks_needed"])
            msg += f"\n\nCarrier delivery location: {where}"
            return {"outcome": "resolved", "recommended_action": "Complete checks", "message": msg}

        if waited:
            msg = pick(RESPONSES["missing_escalate"])
            msg += f"\n\nCarrier delivery location: {where}"
            return {
                "outcome": "needs_escalation",
                "recommended_action": "Open missing-package case",
                "message": msg,
                "ticket_prefix": "MISSING",
            }

        msg = pick(RESPONSES["missing_wait_24h"])
        msg += f"\n\nCarrier delivery location: {where}"
        return {"outcome": "resolved", "recommended_action": "Wait up to 24 hours", "message": msg}

    if issue_key == "4":  # Return order
        if order.status.lower() != "delivered":
            return {
                "outcome": "cannot_process",
                "recommended_action": "Wait for delivery",
                "message": pick(RESPONSES["return_not_delivered"]),
            }

        eligible_until = order.return_eligible_until
        if not eligible_until or now().date() > eligible_until.date():
            return {
                "outcome": "cannot_process",
                "recommended_action": "Contact support for exceptions",
                "message": pick(RESPONSES["return_expired"]),
            }

        # Uses gathered info (reason/opened) in proposed solution
        reason = ctx.get("return_reason", "unspecified")
        opened = "Yes" if ctx.get("opened") else "No"
        msg = pick(RESPONSES["return_created"])
        msg += f"\n\nReturn details:\n- Reason: {reason}\n- Opened/used: {opened}"
        return {"outcome": "resolved", "recommended_action": "Create RMA", "message": msg}

    if issue_key == "5":  # Change address
        if order.status.lower() in {"out for delivery", "delivered"}:
            return {
                "outcome": "cannot_process",
                "recommended_action": "Contact carrier / return after delivery",
                "message": pick(RESPONSES["address_change_blocked"]),
            }

        new_addr = ctx.get("new_address", "").strip()
        msg = pick(RESPONSES["address_change_ok"]) + f"\n\nNew address:\n- {new_addr}"
        return {"outcome": "resolved", "recommended_action": "Apply address change", "message": msg}

    if issue_key == "6":  # Cancel
        if order.status.lower() in {"shipped", "out for delivery", "delivered"}:
            return {
                "outcome": "cannot_process",
                "recommended_action": "Return after delivery",
                "message": pick(RESPONSES["cancel_blocked"]),
            }

        if not ctx.get("confirm_cancel"):
            return {"outcome": "resolved", "recommended_action": "No action", "message": pick(RESPONSES["no_changes"])}

        return {"outcome": "resolved", "recommended_action": "Cancel order", "message": pick(RESPONSES["cancelled"])}

    return {"outcome": "cannot_process", "recommended_action": "Exit", "message": "Unknown request."}


# -----------------------------
# Apply solution (simulation)
# -----------------------------
def apply_solution(issue_key: str, order: Optional[Order], ctx: Dict[str, Any], sol: Dict[str, Any]) -> None:
    if not ctx.get("found_order", False) or order is None:
        print(pick(RESPONSES["no_changes"]))
        print()
        return

    action = sol.get("recommended_action", "")

    if issue_key == "6" and action == "Cancel order":
        order.status = "Cancelled"
        print(pick(RESPONSES["cancelled"]))
        print()
        return

    if issue_key == "5" and action == "Apply address change":
        print(pick(RESPONSES["address_change_ok"]))
        print()
        return

    if issue_key == "4" and action == "Create RMA":
        rma = make_ticket("RMA", order.order_id)
        print(pick(RESPONSES["return_created"]))
        print(f"RMA: {rma}")
        print()
        return

    if issue_key == "2" and action == "Enable proactive updates":
        # no state change needed in this demo
        print("Okay—I'll keep an eye on it and you can check back anytime. (simulated)")
        print()
        return

    print(pick(RESPONSES["no_changes"]))
    print()


# -----------------------------
# Escalation (policy-driven)
# -----------------------------
def escalate_if_needed(issue_key: str, order: Optional[Order], ctx: Dict[str, Any], sol: Dict[str, Any]) -> None:
    if sol.get("outcome") != "needs_escalation":
        return

    order_id = ctx.get("order_id", "UNKNOWN")
    prefix = sol.get("ticket_prefix", "ESC")
    ticket = make_ticket(prefix, order_id)

    # Print a randomized escalation acknowledgment, plus structured details
    print(pick(RESPONSES["escalation_created"]))
    print(f"Case ID: {ticket}")

    if not ctx.get("found_order", False):
        print(f"- Name: {ctx.get('customer_name')}")
        print(f"- Email: {ctx.get('email')}")
        if ctx.get("delivery_zip"):
            print(f"- Delivery ZIP: {ctx.get('delivery_zip')}")
        print()
        return

    assert order is not None
    print(f"- Carrier: {order.carrier}")
    print(f"- Tracking: {order.tracking_number}")

    # Add issue-specific captured context so escalation meaningfully uses prior output
    if issue_key == "2":
        print(f"- Movement in last 48h: {'Yes' if ctx.get('package_moved_recently') else 'No'}")
        notes = ctx.get("delivery_notes", "").strip()
        if notes:
            print(f"- Delivery notes: {notes}")

    if issue_key == "3":
        print(f"- Delivered to: {ctx.get('where_delivered', 'unknown')}")

    print()


# -----------------------------
# Main loop
# -----------------------------
def main() -> None:
    orders = seed_orders()

    while True:
        issue_key = classify_issue()
        if issue_key == "0":
            print("Goodbye! 👋")
            break

        order, ctx = gather_info(issue_key, orders)
        sol = propose_solution(issue_key, order, ctx)

        print(sol["message"])
        print(f"\nRecommended action: {sol['recommended_action']}\n")

        if prompt_yes_no("Proceed? (y/n): "):
            apply_solution(issue_key, order, ctx, sol)
        else:
            print(pick(RESPONSES["no_changes"]))
            print()

        escalate_if_needed(issue_key, order, ctx, sol)


if __name__ == "__main__":
    main()

Welcome. What do you need help with?
Select a support topic:
1) Track order
2) Delivery delayed
3) Missing order
4) Return order
5) Change delivery address
6) Cancel order
0) Exit
Select: 3
Enter your Order ID (e.g., A1001): A1003

--- Order Summary ---
Order ID:         A1003
Name:             Avery Chen
Status:           Delivered
Carrier:          USPS
Tracking Number:  9400111899223847182736
ETA:              2026-03-04 06:14
Delivered At:     2026-03-04 03:14
Return Eligible:  Until 2026-03-19
---------------------

Where does the carrier say it was delivered (porch, mailbox, front desk)? mailbox
Have you checked nearby areas / neighbors / mail room? (y/n): y
Has it been at least 24 hours since it was marked delivered? (y/n): y
You’ve checked and waited 24 hours—let’s open a missing-package case.

Carrier delivery location: mailbox

Recommended action: Open missing-package case

Proceed? (y/n): y
No problem. I won’t make any changes.

I’ve created a case and passed the details to 